In [15]:
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

langlst = ["ar", "ko", "te"]
df_train_filtered = df_train[df_train["lang"].isin(langlst)]
df_validation_filtered = df_validation[df_validation["lang"].isin(langlst)]

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
    # "bert-base-multilingual-cased"
)



df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_te = df_train[(df_train['lang'] == "te")]








## Models chosen:

- 1 n-gram (ask about degrees of n) 

- 2

In [ ]:
from nltk import trigrams
from collections import defaultdict
from collections import Counter

# Tokenize all questions
def tokenizeQuestion(df):
    questions = df["question"]
    tokenized_questions = []

    for question in questions:
        tokenized_questions.append(tokenizer.tokenize(question))

    return tokenized_questions

t_questions = tokenizeQuestion(df_train_ko)

# Build vocabulary for tokens
def buildVocabulary(tokens): 
    tokens_flat = [x for sublist in tokens for x in sublist]
    vocab = Counter(tokens_flat)

    return dict(vocab)

vocabulary = buildVocabulary(t_questions)

t_questions

[['30', '##년', '전쟁', '##의', '승', '##자는', '누', '##구', '##인', '##가', '?'],
 ['엑',
  '##스',
  '##선',
  '##은',
  '누',
  '##가',
  '발',
  '##견',
  '##하',
  '##였',
  '##는',
  '##가',
  '?'],
 ['아',
  '##테',
  '##네',
  '##에서',
  '언',
  '##제',
  '가장',
  '최',
  '##근',
  '##의',
  '올림픽',
  '##이',
  '올',
  '##렸',
  '##나',
  '##요',
  '?'],
 ['세',
  '##상',
  '##에서',
  '가장',
  '오',
  '##래',
  '##된',
  '방송',
  '##사는',
  '무',
  '##엇',
  '##인',
  '##가',
  '?'],
 ['팔', '##레스', '##타', '##인', '수도', '##는', '어', '##딘', '##가', '##요', '?'],
 ['별',
  '##자',
  '##리',
  '중',
  '가장',
  '많은',
  '별',
  '##로',
  '이',
  '##루',
  '##어진',
  '별',
  '##자',
  '##리는',
  '무',
  '##엇',
  '##인',
  '##가',
  '?'],
 ['세',
  '##상',
  '##에서',
  '가장',
  '큰',
  '풍',
  '##력',
  '에',
  '##너',
  '##지',
  '발',
  '##전',
  '##소',
  '##는',
  '무',
  '##엇',
  '##인',
  '##가',
  '?'],
 ['루', '##이', '15', '##세의', '본', '##명은', '무', '##엇', '##인', '##가', '?'],
 ['컬',
  '##러',
  '텔레비전',
  '##이',
  '처음',
  '출',
  '##시',
  '##된',
  '기',
  '##업',
  '##은'

In [154]:
import random
import math



def n_gram_split(sentence, n=1):
    if n < 1:
        raise Exception("n greater or equal to 1")

    grams = []
    for i in range((len(sentence))-(n-1)):
        grams.append([])
        for j in range(n):
            grams[i].append(sentence[i+j])

    return grams

n_grammed_tokens = [n_gram_split(sentence, 3) for sentence in t_questions]
n_grammed_tokens




class Trigram():
    def __init__(self, seed=0):
        self.trigram_count = Counter()
        self.bigram_count = Counter()
        self.vocabulary = set()
        random.seed(seed)


    def train(self, sentences):
        for sentence in sentences:
            tokens = ["<s>", "<s>"] + sentence + ["</s>"]
            self.vocabulary.update(tokens)

            for i in range(2, len(tokens)):
                context = (tokens[i-2], tokens[i-1])
                token = tokens[i]

                self.trigram_count[(context, token)] += 1
                self.bigram_count[context] += 1


    def next_word(self, t1, t2):
        context = (t1, t2)
        total = self.bigram_count[context]

        if total == 0:
            return None

        candidates = []
        probabilities = []


        for (ctx, token), count in self.trigram_count.items():
            if ctx == context:
                candidates.append(token)
                probabilities.append(count / total)

        return random.choices(candidates, weights=probabilities)[0]

    
    def generate(self, max_tokens = 20):
        t1, t2 = "<s>", "<s>"
        result = []

        for _ in range(max_tokens):
            token = self.next_word(t1, t2)

            if token is None or token == "</s>":
                break

            result.append(token)
            t1, t2 = t2, token

        # return " ".join(result).replace(" ##", "")
        return " ".join(result)


    def probability(self, token, t1, t2, alpha=0):
        context = (t1, t2)

        numerator = self.trigram_count[(context, token)] + alpha
        denominator = self.bigram_count[context] + (alpha * len(self.vocabulary))

        if denominator == 0:
            return 0

        return numerator / denominator

    # used for calculating the perplexity
    def perplexity(self, sentences, alpha=0):
        log_prob = 0
        total_tokens = 0

        for sentence in sentences:
            tokens = ["<s>","<s>"] + sentence + ["</s>"]

            for i in range(2, len(tokens)):
                t1 = tokens[i-2]
                t2 = tokens[i-1]
                token = tokens[i]

                p = self.probability(token, t1, t2, alpha)

                if p == 0:
                    return float("inf")

                log_prob += math.log(p)
                total_tokens += 1
        
        if total_tokens == 0:
            return float("inf")

        return math.exp(-log_prob / total_tokens)


lm = Trigram(seed = 3)
lm.train(t_questions)

print(lm.perplexity([["스", "##위", "##스", "사", "##망", "##일", "##은", "언", "##제", "시", "##작", "##되", "##었", "##는", "##가", "?"]]))

print(lm.generate())


2.3714641695949505
스 ##위 ##스 사 ##망 ##일 ##은 언 ##제 시 ##작 ##되 ##었 ##는 ##가 ?


In [ ]:
def uniform_LM():
    return

In [145]:
lst = ["##a", "##b", "##c"]

lst2 = [token.replace("##", "") for token in lst]

lst2

['a', 'b', 'c']